# Практические задачи: очистка и подготовка данных
Задачи к лекции 5 «Очистка и подготовка данных» (`5.da.fillna.rus.pdf`): пропущенные значения, `dropna` и `fillna`, дубликаты, типы данных, выбросы, очистка строк и проверка согласованности.

Сложность: ⭐ — базовая, ⭐⭐ — средняя, ⭐⭐⭐ — повышенная.

## 0. Данные для практики

Во всех заданиях (если не сказано иное) используется таблица заказов интернет-магазина `df_raw`.

| Столбец | Значение |
| :--- | :--- |
| `order_id` | Номер заказа (должен быть уникальным) |
| `mijoz` | Имя клиента |
| `telefon` | Телефон клиента |
| `shahar` | Город доставки |
| `kategoriya` | Категория товара |
| `narx` | Цена за единицу, сум |
| `soni` | Количество |
| `summa` | Сумма заказа (`narx × soni`) |
| `buyurtma_sana` | Дата заказа |
| `yetkazish_sana` | Дата доставки |

Таблица специально «испорчена»: в ней есть почти все проблемы, разобранные на лекции.

Запустите ячейку ниже. **Каждое задание начинайте с `df = df_raw.copy()`**, если в нём не сказано работать с результатом предыдущего задания.

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004, 1004, 1005, 1006, 1007,
                 1008, 1009, 1009, 1010, 1011, 1012, 1013],
    'mijoz': ['  Алишер Каримов ', 'дилноза юсупова', 'Бобур Азимов', 'Камола Рахимова',
              'Камола Рахимова', None, 'Жасур Тошев', 'Нодира Исмоилова',
              'Шахзод Ахмедов', 'Малика Саидова', 'Малика Саидова', 'алишер каримов',
              'Сардор Нуриев', 'Зарина Холова', 'Отабек Юлдашев'],
    'telefon': ['+998 90 123-45-67', '998911234567', '90 555 44 33', '93-111-22-33',
                '93-111-22-33', '+998(94)7778899', '', '97 000 11 22',
                '12-34', '+998 99 888 77 66', '+998 99 888 77 66', '901234567',
                '+998 91 222 33 44', '95-333-44-55', '+998901112233'],
    'shahar': ['Ташкент', 'ташкент ', 'Самарканд', 'Бухара', 'Бухара', 'Tashkent',
               'г. Самарканд', 'N/A', 'Бухара', 'Самарканд', 'Самарканд', 'ТАШКЕНТ',
               'Наманган', '-', 'Наманган'],
    'kategoriya': ['Продукты', 'Электроника', 'Продукты', 'Одежда', 'Одежда', 'Продукты',
                   'Электроника', 'Одежда', 'Продукты', 'Одежда', 'Одежда', 'Электроника',
                   'Продукты', 'Одежда', 'Электроника'],
    'narx': ['12 000', '450000', '8000', '95000', '95000', '?', '520 000 сум', '120000',
             '1500000', '75000', '75000', '380000', '6 500', '110000', ''],
    'soni': [2, 1, np.nan, 1, 1, 3, 1, -2, 1, 2, 2, 1, 4, np.nan, 2],
    'summa': [24000, 450000, np.nan, 95000, 95000, np.nan, 520000, -240000,
              1500000, 150000, 150000, 400000, 26000, np.nan, np.nan],
    'buyurtma_sana': ['2026-09-01', '2026-09-01', '2026-09-02', '2026-09-02', '2026-09-02',
                      '2026-09-03', '2026-09-03', '2026-09-04', '2026-09-04', '2026-09-05',
                      '2026-09-05', '2026-09-06', '2026-09-06', 'нет даты', '2026-09-07'],
    'yetkazish_sana': ['2026-09-02', '2026-09-04', '2026-09-03', '2026-09-05', '2026-09-05',
                       '2026-09-04', '2026-09-02', '2026-09-06', '2026-09-05', None,
                       '2026-09-07', '2026-09-08', '2026-09-07', None, '2026-09-09'],
})
df = df_raw.copy()
display(df.head())


## 1. Поиск пропущенных значений

**Задача 1.1** ⭐ Выведите информацию о таблице с помощью `info()`. В каких столбцах есть пропуски по мнению pandas?

**Задача 1.2** ⭐ Посчитайте количество пропусков в каждом столбце и долю пропусков в процентах (округлите до 1 знака). Выведите результат одной таблицей с двумя столбцами: `kol_vo` и `dolya_%`.

**Задача 1.3** ⭐ Выведите все строки, в которых есть хотя бы один пропуск. Сколько их?

**Задача 1.4** ⭐⭐ Найдите **скрытые пропуски**. Выведите `value_counts(dropna=False)` для столбцов `shahar` и `narx`. Какие значения на самом деле означают «нет данных»?

**Задача 1.5** ⭐⭐ Замените все скрытые пропуски (`'N/A'`, `'-'`, `'?'`, `''`) на `np.nan`. Сравните количество пропусков **до** и **после** замены в одной таблице.

In [ ]:
# Ваш код здесь
# 1.1
df_raw.info()
print("Пропуски по столбцам:")
print(df_raw.isna().sum())

# 1.2
missing = pd.DataFrame({
    "kol_vo": df_raw.isna().sum(),
    "dolya_%": (df_raw.isna().mean() * 100).round(1)
})
display(missing)

# 1.3
rows_with_missing = df_raw[df_raw.isna().any(axis=1)]
display(rows_with_missing)
print("Строк хотя бы с одним пропуском:", len(rows_with_missing))

# 1.4
print("Города:")
print(df_raw["shahar"].value_counts(dropna=False))
print("\nЦены:")
print(df_raw["narx"].value_counts(dropna=False))
print("Скрытые пропуски: 'N/A', '-', '?', пустая строка.")

# 1.5
hidden_missing = {"N/A": np.nan, "-": np.nan, "?": np.nan, "": np.nan}
before = df_raw.isna().sum()
df_hidden = df_raw.replace(hidden_missing)
after = df_hidden.isna().sum()
display(pd.DataFrame({"до": before, "после": after, "добавлено": after-before}))


## 2. Стратегии работы с пропусками: dropna и fillna

Перед заданиями выполните замену скрытых пропусков из задачи 1.5:
```python
df = df_raw.replace({'N/A': np.nan, '-': np.nan, '?': np.nan, '': np.nan})
```

**Задача 2.1** ⭐ Сколько строк останется после каждого вызова? Выведите `shape` для:
* `df.dropna()`
* `df.dropna(subset=['soni'])`
* `df.dropna(how='all')`
* `df.dropna(thresh=9)`

Какой вариант потерял больше всего данных и почему?

**Задача 2.2** ⭐ Заполните пропуски в столбце `mijoz` значением `'Неизвестно'`, а в столбце `shahar` — **модой** (самым частым городом).

**Задача 2.3** ⭐⭐ Для столбца `soni` посчитайте среднее и медиану. Заполните пропуски медианой. Затем сравните среднее значение `soni` в трёх вариантах: пропуски удалены, пропуски заполнены нулём, пропуски заполнены медианой. Почему заполнение нулём — плохая идея для количества?

**Задача 2.4** ⭐⭐ Перед заполнением `soni` создайте столбец-флаг `soni_nan`, который показывает, было ли значение пропущено. Выведите строки, где флаг равен `True`.

**Задача 2.5** ⭐⭐ Дан ряд температур за неделю:
```python
temp = pd.Series([18.0, np.nan, 22.0, np.nan, np.nan, 28.0, np.nan],
                 index=pd.date_range('2026-09-01', periods=7))
```
Заполните пропуски тремя способами: `ffill()`, `bfill()` и `interpolate()`. Выведите результаты рядом в одной таблице. Почему последнее значение после `bfill()` осталось пустым?

**Задача 2.6** ⭐⭐⭐ Дана таблица товаров:
```python
tovar = pd.DataFrame({
    'kategoriya': ['Продукты', 'Продукты', 'Продукты', 'Одежда', 'Одежда', 'Одежда', 'Электроника', 'Электроника'],
    'narx': [5000, np.nan, 9000, 80000, np.nan, 120000, 400000, np.nan],
})
```
Заполните пропущенные цены **медианой своей категории** (через `groupby(...).transform('median')`). Сравните с заполнением общей медианой. Какой вариант разумнее?

In [ ]:
df = df_raw.replace({"N/A": np.nan, "-": np.nan, "?": np.nan, "": np.nan})

# 2.1
print("dropna():", df.dropna().shape)
print("dropna(subset=['soni']):", df.dropna(subset=["soni"]).shape)
print("dropna(how='all'):", df.dropna(how="all").shape)
print("dropna(thresh=9):", df.dropna(thresh=9).shape)
print("Больше всего строк удаляет dropna(): требует отсутствия пропусков во всех столбцах.")

# 2.2
df_fill = df.copy()
df_fill["mijoz"] = df_fill["mijoz"].fillna("Неизвестно")
mode_city = df_fill["shahar"].mode(dropna=True)
if not mode_city.empty:
    df_fill["shahar"] = df_fill["shahar"].fillna(mode_city.iloc[0])
display(df_fill[["mijoz", "shahar"]].head(10))

# 2.3
s = pd.to_numeric(df["soni"], errors="coerce")
print("Среднее:", s.mean(), "Медиана:", s.median())
print("Среднее после удаления пропусков:", s.dropna().mean())
print("Среднее после заполнения нулём:", s.fillna(0).mean())
print("Среднее после заполнения медианой:", s.fillna(s.median()).mean())
print("Ноль может означать отсутствие товара, а не неизвестное количество.")

# 2.4
df_flag = df.copy()
df_flag["soni_nan"] = df_flag["soni"].isna()
display(df_flag[df_flag["soni_nan"]])
df_flag["soni"] = df_flag["soni"].fillna(df_flag["soni"].median())

# 2.5
temp = pd.Series([18.0, np.nan, 22.0, np.nan, np.nan, 28.0, np.nan],
                 index=pd.date_range("2026-09-01", periods=7))
temp_results = pd.DataFrame({
    "исходная": temp,
    "ffill": temp.ffill(),
    "bfill": temp.bfill(),
    "interpolate": temp.interpolate()
})
display(temp_results)
print("Последнее значение bfill остаётся NaN, потому что справа от него нет следующего известного значения.")

# 2.6
tovar = pd.DataFrame({
    "kategoriya": ["Продукты", "Продукты", "Продукты", "Одежда", "Одежда", "Одежда",
                   "Электроника", "Электроника"],
    "narx": [5000, np.nan, 9000, 80000, np.nan, 120000, 400000, np.nan],
})
tovar["narx_median_category"] = tovar["narx"].fillna(
    tovar.groupby("kategoriya")["narx"].transform("median")
)
tovar["narx_median_all"] = tovar["narx"].fillna(tovar["narx"].median())
display(tovar)
print("Медиана категории обычно разумнее: цены разных категорий сильно отличаются.")


## 3. Дублирующиеся строки

**Задача 3.1** ⭐ Сколько в таблице `df_raw` полностью одинаковых строк? Выведите **все** копии (включая первые вхождения).

**Задача 3.2** ⭐ Проверьте, уникален ли столбец `order_id`. Сколько номеров заказов повторяется?

**Задача 3.3** ⭐⭐ Заказ `1009` встречается дважды, но строки не полностью одинаковые. Чем они отличаются? Удалите дубликаты по `order_id` так, чтобы осталась **последняя** (самая свежая) запись. Перенумеруйте индекс. Сколько строк осталось?

**Задача 3.4** ⭐⭐⭐ Клиенты `'  Алишер Каримов '` и `'алишер каримов'` — один и тот же человек. Сколько **уникальных** клиентов в столбце `mijoz` до и после очистки имён (`strip` + `title`)? Выведите список клиентов, которые сделали больше одного заказа (после удаления дубликатов заказов из задачи 3.3).

In [ ]:
# Ваш код здесь
# 3.1
dups_all = df_raw[df_raw.duplicated(keep=False)]
display(dups_all)
print("Полностью дублирующихся строк (включая первые вхождения):", len(dups_all))

# 3.2
print("order_id уникален:", df_raw["order_id"].is_unique)
print("Количество повторных строк по order_id:",
      int(df_raw["order_id"].duplicated(keep=False).sum()))
print("Количество номеров заказов, которые повторяются:",
      int((df_raw["order_id"].value_counts() > 1).sum()))

# 3.3
display(df_raw[df_raw["order_id"] == 1009])
df_dedup = df_raw.drop_duplicates(subset="order_id", keep="last").reset_index(drop=True)
print("Строк после удаления дубликатов заказов:", len(df_dedup))

# 3.4
print("Уникальных имён до очистки:", df_dedup["mijoz"].nunique())
names_clean = df_dedup["mijoz"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
print("Уникальных имён после очистки:", names_clean.nunique())
df_dedup["mijoz_clean"] = names_clean
repeat_customers = df_dedup[df_dedup["mijoz_clean"].notna()].groupby("mijoz_clean")["order_id"].nunique()
print("Клиенты с более чем одним заказом:")
display(repeat_customers[repeat_customers > 1].rename("количество_заказов").reset_index())


## 4. Исправление типов данных

**Задача 4.1** ⭐ Выведите типы всех столбцов `df_raw`. Какие столбцы имеют неправильный тип и какой тип должен быть у них?

**Задача 4.2** ⭐ Попробуйте выполнить `df_raw['narx'].astype(int)`. Какая ошибка возникла и почему? (Оберните код в `try / except`, чтобы ноутбук не остановился.)

**Задача 4.3** ⭐⭐ Преобразуйте `narx` в число с помощью `pd.to_numeric(..., errors='coerce')` **без предварительной очистки**. Выведите исходные значения, которые превратились в `NaN`. Какие из них были на самом деле корректными ценами?

**Задача 4.4** ⭐⭐ Теперь сначала очистите `narx`: удалите все символы, кроме цифр (подсказка: `.str.replace(r'\D', '', regex=True)`), затем преобразуйте в число. Сколько значений теперь стало `NaN`?

**Задача 4.5** ⭐⭐ Преобразуйте `buyurtma_sana` и `yetkazish_sana` в даты, **явно указав формат** `'%Y-%m-%d'`. Какие значения не удалось распознать? Выведите минимальную и максимальную дату заказа.

**Задача 4.6** ⭐⭐ Столбец `soni` имеет тип `float64`, хотя количество — целое число. Преобразуйте его в тип `Int64`, сохранив пропуски.

**Задача 4.7** ⭐⭐ Преобразуйте `kategoriya` в тип `category`. Сравните объём памяти до и после для ряда `pd.Series(df_raw['kategoriya'].tolist() * 10000)` (используйте `memory_usage(deep=True)`).

**Задача 4.8** ⭐ Дан ряд ответов на вопрос «Нужна ли доставка?»:
```python
otvet = pd.Series(['ha', "Yo'q", 'HA', 'yes', 'no', 'Ha '])
```
Преобразуйте его в `True` / `False`.

In [ ]:
# Ваш код здесь
# 4.1
print(df_raw.dtypes)
print("narx — строки, хотя цена должна быть числом; даты пока строки; soni — float из-за NaN.")

# 4.2
try:
    print(df_raw["narx"].astype(int))
except (ValueError, TypeError) as e:
    print("Ошибка:", repr(e))
    print("Причина: в столбце есть текстовые значения, пробелы и пустые/некорректные цены.")

# 4.3
price_coerce = pd.to_numeric(df_raw["narx"], errors="coerce")
display(pd.DataFrame({"исходное": df_raw["narx"], "после_to_numeric": price_coerce})[
    df_raw["narx"].notna() & price_coerce.isna()
])
print("Например, '12 000' и '520 000 сум' — корректные цены, которые не распознаны без очистки.")

# 4.4
price_clean = pd.to_numeric(
    df_raw["narx"].astype("string").str.replace(r"\D", "", regex=True),
    errors="coerce"
)
print("NaN после очистки цены:", int(price_clean.isna().sum()))
display(pd.DataFrame({"исходное": df_raw["narx"], "цена_число": price_clean}))

# 4.5
for col in ["buyurtma_sana", "yetkazish_sana"]:
    converted = pd.to_datetime(df_raw[col], format="%Y-%m-%d", errors="coerce")
    print(f"Не распознаны в {col}:")
    display(df_raw.loc[converted.isna() & df_raw[col].notna(), [col]])
    df_raw_dates = df_raw.copy()
    df_raw_dates[col] = converted
orders_dates = pd.to_datetime(df_raw["buyurtma_sana"], format="%Y-%m-%d", errors="coerce")
print("Минимальная дата заказа:", orders_dates.min())
print("Максимальная дата заказа:", orders_dates.max())

# 4.6
df_types = df_raw.copy()
df_types["soni"] = df_types["soni"].astype("Int64")
print(df_types["soni"].dtype)

# 4.7
cat_series = pd.Series(df_raw["kategoriya"].tolist() * 10000)
before_mem = cat_series.memory_usage(deep=True)
cat_series = cat_series.astype("category")
after_mem = cat_series.memory_usage(deep=True)
print("Память до category:", before_mem, "байт")
print("Память после category:", after_mem, "байт")

# 4.8
otvet = pd.Series(["ha", "Yo'q", "HA", "yes", "no", "Ha "])
normalized = otvet.astype("string").str.strip().str.lower()
answer_map = {"ha": True, "yes": True, "yo'q": False, "no": False}
otvet_bool = normalized.map(answer_map)
display(pd.DataFrame({"исходный": otvet, "bool": otvet_bool}))


## 5. Выбросы (outliers)

Для заданий этого раздела используйте очищенную цену:
```python
narx = pd.to_numeric(df_raw['narx'].str.replace(r'\D', '', regex=True), errors='coerce').dropna()
```

**Задача 5.1** ⭐ Выведите `describe()` для `narx`. Сравните среднее и медиану. Что подсказывает разница между `75%` и `max`?

**Задача 5.2** ⭐⭐ Напишите функцию `iqr_bounds(s, k=1.5)`, которая возвращает нижнюю и верхнюю границы по методу IQR. Найдите выбросы в `narx`.

**Задача 5.3** ⭐⭐ Найдите выбросы в `narx` по z-оценке (`|z| > 3`) и по робастной z-оценке (через медиану и MAD, порог 3.5). Выведите максимальное значение `|z|` для обоих методов. Насколько «уверенно» каждый метод нашёл выброс? Объясните разницу.

**Задача 5.4** ⭐⭐⭐ Цены сильно зависят от категории: 1 500 000 — нормально для электроники, но не для продуктов. Найдите выбросы по IQR **внутри каждой категории** (подсказка: `groupby('kategoriya')['narx'].transform(lambda s: s.quantile(0.25))`). Какой заказ оказался выбросом?

**Задача 5.5** ⭐⭐ Даны баллы за экзамен (максимум — 100):
```python
ball = pd.Series([72, 85, 64, 91, 0, 78, 150, 88, 69, 100, 95, -5])
```
Найдите невозможные значения по **бизнес-правилу** (балл от 0 до 100). Затем ограничьте значения диапазоном `[0, 100]` с помощью `clip`. Подумайте: является ли балл `0` ошибкой?

In [ ]:
# Ваш код здесь
narx = pd.to_numeric(
    df_raw["narx"].astype("string").str.replace(r"\D", "", regex=True),
    errors="coerce"
).dropna()

# 5.1
display(narx.describe())
print("Среднее:", narx.mean(), "Медиана:", narx.median())
print("Большой разрыв между 75% и max может указывать на высокую цену-выброс.")

# 5.2
def iqr_bounds(s, k=1.5):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

low, high = iqr_bounds(narx)
print("Границы IQR:", low, high)
display(narx[(narx < low) | (narx > high)])

# 5.3
z = (narx - narx.mean()) / narx.std(ddof=0)
mad = (narx - narx.median()).abs().median()
robust_z = 0.6745 * (narx - narx.median()) / mad if mad != 0 else pd.Series(np.nan, index=narx.index)
print("Выбросы по |z| > 3:")
display(narx[z.abs() > 3])
print("Максимальный |z|:", z.abs().max())
print("Выбросы по робастной z-оценке > 3.5:")
display(narx[robust_z.abs() > 3.5])
print("Максимальный |robust z|:", robust_z.abs().max())
print("Оценки могут отличаться, потому что обычная z-оценка использует среднее и стандартное отклонение, чувствительные к выбросам.")

# 5.4 — IQR внутри категории
out_df = df_raw.copy()
out_df["narx_num"] = pd.to_numeric(
    out_df["narx"].astype("string").str.replace(r"\D", "", regex=True), errors="coerce"
)
q1 = out_df.groupby("kategoriya")["narx_num"].transform(lambda s: s.quantile(0.25))
q3 = out_df.groupby("kategoriya")["narx_num"].transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
out_df["outlier_category"] = (out_df["narx_num"] < q1 - 1.5*iqr) | (out_df["narx_num"] > q3 + 1.5*iqr)
display(out_df.loc[out_df["outlier_category"], ["order_id", "kategoriya", "narx_num"]])

# 5.5
ball = pd.Series([72, 85, 64, 91, 0, 78, 150, 88, 69, 100, 95, -5])
print("Невозможные по правилу [0, 100]:")
display(ball[(ball < 0) | (ball > 100)])
ball_clipped = ball.clip(0, 100)
print("После clip:")
print(ball_clipped.tolist())
print("0 не обязательно ошибка: это может быть реальный нулевой балл.")


## 6. Очистка строк: strip, replace, regex

**Задача 6.1** ⭐ Приведите столбец `mijoz` к единому виду: уберите лишние пробелы по краям и внутри строки, каждое слово — с заглавной буквы.

**Задача 6.2** ⭐⭐ Нормализуйте столбец `shahar`. В результате должны остаться только значения `Ташкент`, `Самарканд`, `Бухара`, `Наманган` и `NaN`. Выведите `value_counts(dropna=False)` до и после.
Подсказки: `strip` → `lower` → убрать префикс `'г. '` с помощью regex → словарь замен → `title`.

**Задача 6.3** ⭐⭐ Нормализуйте телефоны к формату `+998XXXXXXXXX`:
1. удалите все символы, кроме цифр;
2. если цифр 12 и номер начинается с `998` — оставьте последние 9 цифр; если цифр 9 — оставьте как есть; иначе — `NaN`;
3. добавьте `'+998'` в начало.

Какие номера оказались некорректными?

**Задача 6.4** ⭐⭐ Дан ряд цен в произвольном формате:
```python
price_txt = pd.Series(['12 000 сум', 'цена: 9500', '7,500', '15000 so\'m', 'договорная', '1 200 000'])
```
Извлеките число из каждой строки с помощью `.str.extract`, очистите разделители и преобразуйте в число.

**Задача 6.5** ⭐⭐ Дан ряд email-адресов:
```python
emails = pd.Series(['ali@mail.uz', ' Bobur@Gmail.com', 'kamola.mail.ru', 'test@', 'sardor_n@tiu.uz '])
```
Очистите адреса (пробелы, нижний регистр) и проверьте, соответствует ли каждый адрес формату `имя@домен.зона` с помощью `.str.match`.

**Задача 6.6** ⭐ Дан ряд адресов:
```python
manzil = pd.Series(['Ташкент, Чиланзар, 12', 'Самарканд, Регистан, 5', 'Бухара, Центр, 7'])
```
Разделите его на три столбца: `shahar`, `tuman`, `uy` с помощью `.str.split(..., expand=True)`.

**Задача 6.7** ⭐⭐⭐ Найдите в `df_raw` всех клиентов, в имени которых есть буквосочетание «ов» (без учёта регистра), и всех клиентов, чьё **имя** (первое слово) начинается на «А» или «Б».

In [ ]:
# Ваш код здесь
# 6.1
df_strings = df_raw.copy()
df_strings["mijoz"] = (
    df_strings["mijoz"].astype("string")
    .str.strip().str.replace(r"\s+", " ", regex=True).str.title()
)
display(df_strings[["mijoz"]])

# 6.2
print("Города до нормализации:")
print(df_raw["shahar"].value_counts(dropna=False))
city = df_raw["shahar"].astype("string").str.strip().str.lower()
city = city.str.replace(r"^г\.\s*", "", regex=True)
city_map = {
    "ташкент": "Ташкент", "tashkent": "Ташкент",
    "самарканд": "Самарканд", "бухара": "Бухара", "наманган": "Наманган"
}
city = city.map(city_map)
print("Города после нормализации:")
print(city.value_counts(dropna=False))

# 6.3
def normalize_phone(value):
    if pd.isna(value):
        return np.nan
    digits = "".join(ch for ch in str(value) if ch.isdigit())
    if len(digits) == 12 and digits.startswith("998"):
        digits = digits[-9:]
    elif len(digits) == 9:
        pass
    else:
        return np.nan
    return "+998" + digits

phones = df_raw["telefon"].map(normalize_phone)
display(pd.DataFrame({"исходный": df_raw["telefon"], "нормализованный": phones}))
print("Некорректные телефоны:")
display(df_raw.loc[phones.isna(), ["order_id", "telefon"]])

# 6.4
price_txt = pd.Series(["12 000 сум", "цена: 9500", "7,500", "15000 so'm", "договорная", "1 200 000"])
extracted = price_txt.str.extract(r"([\d\s,]+)", expand=False)
extracted_clean = extracted.str.replace(r"[\s,]", "", regex=True)
price_num = pd.to_numeric(extracted_clean, errors="coerce")
display(pd.DataFrame({"исходный": price_txt, "извлечено": extracted, "число": price_num}))

# 6.5
emails = pd.Series(["ali@mail.uz", " Bobur@Gmail.com", "kamola.mail.ru", "test@", "sardor_n@tiu.uz "])
emails_clean = emails.str.strip().str.lower()
email_valid = emails_clean.str.match(r"^[\w.!#$%&'*+/=?^`{|}~-]+@[a-z0-9-]+(?:\.[a-z0-9-]+)+$", na=False)
display(pd.DataFrame({"email": emails_clean, "валидный": email_valid}))

# 6.6
manzil = pd.Series(["Ташкент, Чиланзар, 12", "Самарканд, Регистан, 5", "Бухара, Центр, 7"])
address_parts = manzil.str.split(r"\s*,\s*", expand=True)
address_parts.columns = ["shahar", "tuman", "uy"]
display(address_parts)

# 6.7
names = df_raw["mijoz"].astype("string")
print("Имена, содержащие «ов» без учёта регистра:")
display(df_raw[names.str.contains("ов", case=False, na=False)])
first_name = names.str.strip().str.split().str[0]
print("Клиенты, чьё первое слово начинается на А или Б:")
display(df_raw[first_name.str.match(r"^[АБ]", case=False, na=False)])


## 7. Проверка согласованности данных

Для заданий этого раздела подготовьте таблицу:
```python
df = df_raw.drop_duplicates(subset='order_id', keep='last').reset_index(drop=True)
df['narx'] = pd.to_numeric(df['narx'].str.replace(r'\D', '', regex=True), errors='coerce')
for col in ['buyurtma_sana', 'yetkazish_sana']:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')
```

**Задача 7.1** ⭐ Найдите заказы с неположительным количеством (`soni <= 0`).

**Задача 7.2** ⭐ Найдите заказы, у которых дата доставки **раньше** даты заказа.

**Задача 7.3** ⭐⭐ Проверьте правило `summa == narx × soni`. Выведите строки, где правило нарушено (строки с пропусками в этих столбцах не учитывайте).

**Задача 7.4** ⭐⭐ Допустимые категории: `Продукты`, `Одежда`, `Электроника`, `Бытовая техника`. Проверьте, что все категории допустимы.

**Задача 7.5** ⭐⭐⭐ Напишите функцию `check(df)`, которая возвращает `pd.Series` с количеством проблем по каждой проверке:
* повторяющиеся `order_id`;
* `soni <= 0`;
* `narx <= 0`;
* `summa` не равна `narx × soni`;
* доставка раньше заказа;
* дата заказа в будущем;
* общее количество пропусков.

Запустите её на `df`.

**Задача 7.6** ⭐⭐ Добавьте проверки `assert`: номера заказов уникальны, количество положительно. Убедитесь, что на неочищенных данных `assert` срабатывает (оберните в `try / except AssertionError`).

In [ ]:
# Ваш код здесь
df = df_raw.drop_duplicates(subset="order_id", keep="last").reset_index(drop=True)
df["narx"] = pd.to_numeric(df["narx"].astype("string").str.replace(r"\D", "", regex=True), errors="coerce")
for col in ["buyurtma_sana", "yetkazish_sana"]:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")

# 7.1
print("Заказы с soni <= 0:")
display(df[df["soni"].le(0).fillna(False)])

# 7.2
print("Доставка раньше заказа:")
display(df[df["yetkazish_sana"].notna() & df["buyurtma_sana"].notna() &
           (df["yetkazish_sana"] < df["buyurtma_sana"])])

# 7.3
valid = df[["summa", "narx", "soni"]].notna().all(axis=1)
mismatch = valid & ~np.isclose(df["summa"], df["narx"] * df["soni"])
display(df.loc[mismatch, ["order_id", "narx", "soni", "summa"]])

# 7.4
allowed_categories = {"Продукты", "Одежда", "Электроника", "Бытовая техника"}
invalid_categories = df[~df["kategoriya"].isin(allowed_categories)]
print("Недопустимые категории:")
display(invalid_categories[["order_id", "kategoriya"]])

# 7.5
def check(data):
    dup_ids = int(data["order_id"].duplicated(keep=False).sum())
    nonpositive_qty = int(data["soni"].le(0).fillna(False).sum())
    price = pd.to_numeric(data["narx"].astype("string").str.replace(r"\D", "", regex=True), errors="coerce")
    nonpositive_price = int(price.le(0).fillna(False).sum())
    qty = pd.to_numeric(data["soni"], errors="coerce")
    total = pd.to_numeric(data["summa"], errors="coerce")
    comparable = price.notna() & qty.notna() & total.notna()
    wrong_total = int((comparable & ~np.isclose(total, price * qty)).sum())
    order_date = pd.to_datetime(data["buyurtma_sana"], format="%Y-%m-%d", errors="coerce")
    delivery_date = pd.to_datetime(data["yetkazish_sana"], format="%Y-%m-%d", errors="coerce")
    delivery_before_order = int((delivery_date.notna() & order_date.notna() & (delivery_date < order_date)).sum())
    future_order_date = int((order_date > pd.Timestamp.today().normalize()).sum())
    missing_total = int(data.isna().sum().sum())
    return pd.Series({
        "повторяющиеся order_id (строки)": dup_ids,
        "soni <= 0": nonpositive_qty,
        "narx <= 0": nonpositive_price,
        "summa != narx * soni": wrong_total,
        "доставка раньше заказа": delivery_before_order,
        "дата заказа в будущем": future_order_date,
        "всего пропусков": missing_total
    })

display(check(df))

# 7.6
try:
    assert df_raw["order_id"].is_unique, "Есть повторяющиеся order_id"
    assert df_raw["soni"].dropna().gt(0).all(), "Есть неположительное количество"
except AssertionError as e:
    print("Проверка не пройдена:", e)


## 8. Итоговое задание: конвейер очистки

**Задача 8.1** ⭐⭐⭐ Напишите функцию `clean(raw)`, которая возвращает очищенную копию `df_raw`. Соблюдайте порядок шагов из лекции:

1. **Скрытые пропуски** → `NaN`.
2. **Строки**: `mijoz` (пробелы, регистр), `shahar` (единое написание), `telefon` (формат `+998XXXXXXXXX`).
3. **Типы**: `narx` → число, даты → `datetime`, `kategoriya` → `category`.
4. **Дубликаты**: по `order_id`, оставить последнюю запись.
5. **Ошибки и выбросы**:
   * `soni <= 0` → `NaN`;
   * выбросы цены по IQR **внутри категории** → `NaN`;
   * дата доставки раньше даты заказа → `NaN` в `yetkazish_sana`.
6. **Пропуски**:
   * `narx` — медиана категории;
   * `soni` — медиана, затем тип `int`;
   * `mijoz`, `shahar` — `'Неизвестно'`;
   * `buyurtma_sana` — предыдущая дата (`ffill`); `yetkazish_sana` оставить пустой (заказ мог быть ещё не доставлен).
7. **Пересчитать** `summa = narx × soni`.

**Задача 8.2** ⭐⭐ Запустите `check()` из задачи 7.5 **до** и **после** очистки и выведите результаты рядом.

**Задача 8.3** ⭐⭐ Добавьте в конце `assert`-проверки: `order_id` уникален, `soni > 0`, в `narx` и `summa` нет пропусков.

**Задача 8.4** ⭐⭐ Составьте таблицу «было — стало»: количество строк, пропусков, уникальных городов, уникальных клиентов, средняя и медианная сумма заказа.

**Задача 8.5** ⭐⭐⭐ (Для размышления) Заказ `1008`: продукт за 1 500 000. Мы заменили цену медианой категории. Предложите другой способ обработки и объясните, когда он был бы лучше.

In [ ]:
# Ваш код здесь
def clean(raw):
    data = raw.copy()

    # 1. Скрытые пропуски
    data = data.replace({"N/A": np.nan, "-": np.nan, "?": np.nan, "": np.nan})

    # 2. Очистка строк
    data["mijoz"] = data["mijoz"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()

    city = data["shahar"].astype("string").str.strip().str.lower()
    city = city.str.replace(r"^г\.\s*", "", regex=True)
    city_map = {
        "ташкент": "Ташкент", "tashkent": "Ташкент",
        "самарканд": "Самарканд", "бухара": "Бухара", "наманган": "Наманган"
    }
    data["shahar"] = city.map(city_map)

    data["telefon"] = data["telefon"].map(normalize_phone)

    # 3. Типы
    data["narx"] = pd.to_numeric(
        data["narx"].astype("string").str.replace(r"\D", "", regex=True), errors="coerce"
    )
    for col in ["buyurtma_sana", "yetkazish_sana"]:
        data[col] = pd.to_datetime(data[col], format="%Y-%m-%d", errors="coerce")
    data["kategoriya"] = data["kategoriya"].astype("category")
    data["soni"] = pd.to_numeric(data["soni"], errors="coerce")

    # 4. Удаление дубликатов
    data = data.drop_duplicates(subset="order_id", keep="last").reset_index(drop=True)

    # 5. Ошибки и выбросы
    data.loc[data["soni"] <= 0, "soni"] = np.nan
    q1 = data.groupby("kategoriya", observed=False)["narx"].transform(lambda s: s.quantile(0.25))
    q3 = data.groupby("kategoriya", observed=False)["narx"].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    outliers = (data["narx"] < q1 - 1.5 * iqr) | (data["narx"] > q3 + 1.5 * iqr)
    data.loc[outliers, "narx"] = np.nan

    invalid_delivery = (
        data["yetkazish_sana"].notna() & data["buyurtma_sana"].notna() &
        (data["yetkazish_sana"] < data["buyurtma_sana"])
    )
    data.loc[invalid_delivery, "yetkazish_sana"] = pd.NaT

    # 6. Заполнение пропусков
    category_median = data.groupby("kategoriya", observed=False)["narx"].transform("median")
    data["narx"] = data["narx"].fillna(category_median)
    data["soni"] = data["soni"].fillna(data["soni"].median()).round().astype("Int64")
    data["mijoz"] = data["mijoz"].fillna("Неизвестно")
    data["shahar"] = data["shahar"].fillna("Неизвестно")
    data["buyurtma_sana"] = data["buyurtma_sana"].ffill()

    # 7. Пересчёт суммы
    data["summa"] = data["narx"] * data["soni"]
    return data

clean_df = clean(df_raw)
display(clean_df)


In [ ]:
# 8.2 — сравнение check до и после
before_check = check(df_raw)
after_check = check(clean_df)
display(pd.DataFrame({"до очистки": before_check, "после очистки": after_check}))

# 8.3 — итоговые проверки
assert clean_df["order_id"].is_unique, "order_id должны быть уникальными"
assert clean_df["soni"].gt(0).all(), "Количество должно быть положительным"
assert clean_df["narx"].notna().all(), "В цене не должно быть пропусков"
assert clean_df["summa"].notna().all(), "В сумме не должно быть пропусков"
print("Все итоговые assert-проверки пройдены.")

# 8.4 — таблица «было — стало»
def summary_table(data):
    return pd.Series({
        "количество строк": len(data),
        "количество пропусков": int(data.isna().sum().sum()),
        "уникальных городов": data["shahar"].nunique(dropna=True),
        "уникальных клиентов": data["mijoz"].nunique(dropna=True),
        "средняя сумма заказа": pd.to_numeric(data["summa"], errors="coerce").mean(),
        "медианная сумма заказа": pd.to_numeric(data["summa"], errors="coerce").median()
    })

display(pd.DataFrame({
    "было": summary_table(df_raw),
    "стало": summary_table(clean_df)
}))

# 8.5 — альтернативы для заказа 1008
print("Заказ 1008: альтернативы замене цены медианой — проверить чек/накладную, "
      "сверить цену с источником или оставить цену пропущенной до проверки. "
      "Это лучше, если 1 500 000 может быть реальной ценой дорогого товара.")


## 9. Вопросы для самопроверки

Ответьте письменно, 1–3 предложения на вопрос.

1. Почему `np.nan == np.nan` возвращает `False`? Как правильно проверять пропуски?
2. Что такое скрытые пропуски? Приведите три примера.
3. Чем отличаются MCAR, MAR и MNAR? Приведите пример MNAR.
4. Почему `df.dropna()` без параметров может удалить большую часть таблицы, даже если в каждом столбце мало пропусков?
5. Когда для заполнения пропусков лучше использовать медиану, а не среднее?
6. Почему в pandas 3 запись `df['col'].fillna(0, inplace=True)` не изменяет таблицу?
7. Почему дубликаты нужно искать **после** очистки строк?
8. В чём опасность `errors='coerce'`? Как её избежать?
9. Почему z-оценка может не найти очевидный выброс?
10. Чем отличается `Series.str.replace()` от `Series.replace()`?
11. Приведите два примера межстолбцовых (логических) проверок для таблицы заказов.
12. Зачем использовать `assert` в конце скрипта очистки?

In [ ]:
1. Почему np.nan == np.nan возвращает False? NaN обозначает отсутствующее/неопределённое значение, поэтому сравнение на равенство не работает как для обычных чисел. Для проверки пропусков используют isna() или isnull().
2. Что такое скрытые пропуски? Это значения, которые выглядят как обычный текст, но фактически означают отсутствие данных: '', 'N/A', '-', '?'.
3. MCAR, MAR и MNAR: MCAR — пропуск не зависит от данных; MAR — зависит от наблюдаемых переменных; MNAR — зависит от самого пропущенного значения или ненаблюдаемой причины. Пример MNAR: люди с очень высоким доходом чаще не указывают доход.
4. Почему dropna() может удалить много строк? По умолчанию удаляется любая строка хотя бы с одним пропуском; пропуски в разных столбцах складываются и затрагивают много строк.
5. Когда лучше медиана, а не среднее? Когда данные имеют выбросы или сильно асимметричное распределение, поскольку медиана менее чувствительна к экстремальным значениям.
6. Почему fillna(..., inplace=True) может не изменить таблицу в pandas 3? Из-за Copy-on-Write и поведения промежуточных объектов изменения через выбранный столбец не гарантируют обновление исходного DataFrame. Надёжнее присваивать результат: df['col'] = df['col'].fillna(0).
7. Почему дубликаты ищут после очистки строк? Записи одного клиента могут различаться пробелами, регистром или форматированием, поэтому до нормализации они не выглядят одинаковыми.
8. В чём опасность errors='coerce'? Все нераспознанные значения превращаются в NaN, включая потенциально корректные значения в необычном формате. Нужно отдельно проверять исходные значения, ставшие NaN.
9. Почему z-оценка может не найти выброс? Выброс сам увеличивает среднее и стандартное отклонение, из-за чего его z-оценка становится менее экстремальной.
10. Series.str.replace() и Series.replace(): .str.replace() заменяет текстовые шаблоны внутри строк; .replace() заменяет значения целиком или по заданному соответствию.
11. Два примера межстолбцовых проверок: summa == narx * soni; дата доставки не должна быть раньше даты заказа.
12. Зачем assert в конце очистки? Он автоматически останавливает выполнение с ошибкой, если важное условие качества данных нарушено.